# 🚀 DeepSeek V3 아키텍처 개요

이 노트북에서는 DeepSeek V3의 전체 아키텍처를 개괄적으로 살펴보고, 핵심 개념을 이해하기 위한 테스트를 진행합니다.

**참고 자료:**
- 논문: https://arxiv.org/pdf/2412.19437
- 코드: https://github.com/huggingface/transformers/blob/main/src/transformers/models/deepseek_v3/modeling_deepseek_v3.py


## 1. DeepSeek V3 모델 개요

DeepSeek V3는 **Mixture-of-Experts (MoE)** 아키텍처를 기반으로 한 대규모 언어 모델입니다.

### 📊 주요 스펙

| 항목 | 값 | 설명 |
|------|-----|------|
| 총 파라미터 수 | 671B | 6,710억 개 |
| 활성화 파라미터 수 | 37B | 토큰당 370억 개 |
| 학습 토큰 수 | 14.8T | 14조 8천억 개 |
| 학습 비용 | 2,788만 H800 GPU 시간 | 약 $5.6M |
| 컨텍스트 길이 | 128K 토큰 | 최대 160K 지원 |

### 🔑 핵심 혁신

1. **Multi-head Latent Attention (MLA)** - KV 캐시 압축으로 메모리 효율성 향상
2. **DeepSeekMoE** - Auxiliary loss 없는 로드 밸런싱
3. **Multi-Token Prediction (MTP)** - 다중 토큰 예측으로 추론 가속화


## 📝 이해도 테스트 1: 기본 개념

아래 질문에 답해보세요. 답을 작성한 후 다음 셀을 실행하여 정답을 확인하세요.


In [ ]:
# 질문 1: DeepSeek V3의 총 파라미터 수는?
your_answer_1 = ""  # 예: "671B" 또는 "6710억"

# 질문 2: 토큰당 활성화되는 파라미터 수는?
your_answer_2 = ""  # 예: "37B" 또는 "370억"

# 질문 3: MoE에서 총 전문가 수와 토큰당 활성화 전문가 수는?
your_answer_3_total = 0  # 총 전문가 수
your_answer_3_active = 0  # 활성화 전문가 수


In [ ]:
# 정답 확인
print("📋 정답 확인")
print("=" * 50)

correct_1 = "671B" in your_answer_1 or "6710" in your_answer_1
print(f"질문 1: {'✅ 정답!' if correct_1 else '❌ 오답'} (정답: 671B, 6,710억 개)")

correct_2 = "37B" in your_answer_2 or "370" in your_answer_2
print(f"질문 2: {'✅ 정답!' if correct_2 else '❌ 오답'} (정답: 37B, 370억 개)")

correct_3 = your_answer_3_total == 256 and your_answer_3_active == 8
print(f"질문 3: {'✅ 정답!' if correct_3 else '❌ 오답'} (정답: 총 256개, 활성화 8개)")

print("\n💡 설명:")
print("- 총 671B 파라미터 중 토큰당 37B만 활성화됩니다.")
print("- 이는 MoE 아키텍처 덕분에 가능합니다.")
print("- 256개 전문가 중 8개만 선택되어 활성화됩니다 (약 3.1%).")


## 2. 전체 아키텍처 구조

DeepSeek V3의 전체 구조를 시각화해봅시다.

```
Input Token IDs
      │
      ▼
Token Embedding Layer
      │
      ▼
┌─────────────────────────────────┐
│     Decoder Layer × 61         │
│  ┌───────────────────────────┐ │
│  │ RMSNorm                   │ │
│  │     ↓                     │ │
│  │ MLA Attention             │ │
│  │     ↓ + residual          │ │
│  │ RMSNorm                   │ │
│  │     ↓                     │ │
│  │ MLP or MoE                │ │
│  │     ↓ + residual          │ │
│  └───────────────────────────┘ │
└─────────────────────────────────┘
      │
      ▼
Final RMSNorm
      │
      ▼
LM Head (Linear)
      │
      ▼
Output Logits
```


In [ ]:
import torch
import torch.nn as nn

# DeepSeek V3 설정 (간소화 버전)
class SimpleConfig:
    vocab_size = 102400
    hidden_size = 7168
    num_hidden_layers = 61
    num_attention_heads = 128
    intermediate_size = 18432
    
config = SimpleConfig()

print("📊 DeepSeek V3 기본 설정")
print("=" * 50)
print(f"어휘 크기 (vocab_size): {config.vocab_size:,}")
print(f"히든 차원 (hidden_size): {config.hidden_size:,}")
print(f"레이어 수 (num_hidden_layers): {config.num_hidden_layers}")
print(f"어텐션 헤드 수 (num_attention_heads): {config.num_attention_heads}")
print(f"MLP 중간 차원 (intermediate_size): {config.intermediate_size:,}")


## 📝 이해도 테스트 2: 아키텍처 구조

아래 코드를 완성하여 간단한 Transformer 블록의 데이터 흐름을 구현해보세요.


In [ ]:
# TODO: 아래 빈칸을 채워서 Transformer 블록의 기본 구조를 완성하세요

class SimpleTransformerBlock(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_size)
        self.attn = nn.MultiheadAttention(hidden_size, num_heads=8, batch_first=True)
        self.norm2 = nn.LayerNorm(hidden_size)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_size, hidden_size * 4),
            nn.GELU(),
            nn.Linear(hidden_size * 4, hidden_size)
        )
    
    def forward(self, x):
        # TODO: Pre-norm + Attention + Residual 구현
        # 힌트: h1 = x + attn(norm1(x))
        h1 = None  # 여기를 채우세요
        
        # TODO: Pre-norm + MLP + Residual 구현
        # 힌트: h2 = h1 + mlp(norm2(h1))
        h2 = None  # 여기를 채우세요
        
        return h2

# 테스트
print("테스트를 위해 forward 메서드를 완성하세요!")


In [ ]:
# 정답 코드

class SimpleTransformerBlockAnswer(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_size)
        self.attn = nn.MultiheadAttention(hidden_size, num_heads=8, batch_first=True)
        self.norm2 = nn.LayerNorm(hidden_size)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_size, hidden_size * 4),
            nn.GELU(),
            nn.Linear(hidden_size * 4, hidden_size)
        )
    
    def forward(self, x):
        # Pre-norm + Attention + Residual
        normed = self.norm1(x)
        attn_out, _ = self.attn(normed, normed, normed)
        h1 = x + attn_out  # Residual connection
        
        # Pre-norm + MLP + Residual
        h2 = h1 + self.mlp(self.norm2(h1))  # Residual connection
        
        return h2

# 테스트
hidden_size = 256
block = SimpleTransformerBlockAnswer(hidden_size)

# 입력 생성
batch_size, seq_len = 2, 16
x = torch.randn(batch_size, seq_len, hidden_size)

# Forward pass
output = block(x)

print("✅ 정답 코드 실행 결과:")
print(f"입력 형태: {x.shape}")
print(f"출력 형태: {output.shape}")
print(f"입출력 형태 일치: {x.shape == output.shape}")


## 3. 핵심 수식 정리

### 📐 RMSNorm

$$\text{RMSNorm}(x) = \frac{x}{\sqrt{\text{mean}(x^2) + \epsilon}} \times \gamma$$

### 📐 Multi-head Latent Attention (MLA)

**KV 압축:**
$$c_{KV} = W_{DKV} \cdot h$$

**KV 복원:**
$$k_C = W_{UK} \cdot c_{KV}, \quad v = W_{UV} \cdot c_{KV}$$

**Attention:**
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) \cdot V$$

### 📐 MoE 라우팅

**라우터 출력:**
$$s_i = \text{softmax}(W_{router} \cdot h)_i$$

**최종 출력:**
$$y = \sum_{i \in \text{TopK}} g_i \cdot \text{Expert}_i(h)$$

### 📐 SwiGLU Activation

$$\text{SwiGLU}(x) = (x \cdot W_{gate}) \odot \text{SiLU}(x \cdot W_{up})$$
$$\text{MLP}(x) = W_{down} \cdot \text{SwiGLU}(x)$$


## 📝 이해도 테스트 3: 수식 이해

아래 코드를 실행하여 SwiGLU 활성화 함수를 구현해보세요.


In [ ]:
import torch.nn.functional as F

def silu(x):
    """SiLU (Swish) 활성화 함수: x * sigmoid(x)"""
    return x * torch.sigmoid(x)

class SwiGLU(nn.Module):
    """
    SwiGLU 활성화 함수
    
    수식: SwiGLU(x) = (x @ W_gate) ⊙ SiLU(x @ W_up)
    """
    def __init__(self, hidden_size, intermediate_size):
        super().__init__()
        self.gate_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.up_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.down_proj = nn.Linear(intermediate_size, hidden_size, bias=False)
    
    def forward(self, x):
        # SwiGLU 구현
        gate = self.gate_proj(x)
        up = silu(self.up_proj(x))
        return self.down_proj(gate * up)

# 테스트
hidden_size = 256
intermediate_size = 512

swiglu = SwiGLU(hidden_size, intermediate_size)
x = torch.randn(2, 16, hidden_size)
output = swiglu(x)

print("✅ SwiGLU 테스트 결과:")
print(f"입력 형태: {x.shape}")
print(f"출력 형태: {output.shape}")
print(f"형태 일치: {x.shape == output.shape}")


## 4. 요약 퀴즈

아래 질문에 True/False로 답해보세요.


In [ ]:
# 퀴즈
quiz = {
    "Q1: DeepSeek V3는 모든 파라미터를 항상 사용한다.": None,  # True or False?
    "Q2: MLA는 KV 캐시를 압축하여 메모리를 절약한다.": None,
    "Q3: MoE에서 모든 전문가가 모든 토큰을 처리한다.": None,
    "Q4: RMSNorm은 LayerNorm보다 계산이 효율적이다.": None,
    "Q5: DeepSeek V3의 컨텍스트 길이는 최대 128K이다.": None,
}

# 여기에 답을 입력하세요
quiz["Q1: DeepSeek V3는 모든 파라미터를 항상 사용한다."] = False  # 예시
# quiz["Q2: ..."] = True or False


In [ ]:
# 정답 확인
answers = {
    "Q1: DeepSeek V3는 모든 파라미터를 항상 사용한다.": False,
    "Q2: MLA는 KV 캐시를 압축하여 메모리를 절약한다.": True,
    "Q3: MoE에서 모든 전문가가 모든 토큰을 처리한다.": False,
    "Q4: RMSNorm은 LayerNorm보다 계산이 효율적이다.": True,
    "Q5: DeepSeek V3의 컨텍스트 길이는 최대 128K이다.": True,
}

print("📋 퀴즈 정답 확인")
print("=" * 60)

correct = 0
for q, a in answers.items():
    user_answer = quiz.get(q)
    is_correct = user_answer == a
    if is_correct:
        correct += 1
    status = "✅" if is_correct else "❌"
    print(f"{status} {q}")
    print(f"   정답: {a}, 당신의 답: {user_answer}")
    print()

print(f"\n🎯 점수: {correct}/{len(answers)} ({correct/len(answers)*100:.0f}%)")


## 🎉 다음 단계

이 노트북에서 DeepSeek V3의 전체 아키텍처 개요를 살펴보았습니다.

다음 노트북에서는 **RMSNorm**과 **RoPE (Rotary Position Embedding)**를 자세히 살펴봅니다.

➡️ **02_rmsnorm_rope_tutorial.ipynb**로 이동하세요!
